# DX 601 Week 5 Homework

## Introduction

In this homework, I will give you a random algorithm $A$ and a way to verify whether the output of $A$ is correct or not. Your job is to construct an algorithm that uses algorithm $A$ to always solve the problem algorithm $A$ is trying to solve (i.e. algorithm $A$ only solves it some of the time while your algorithm must solve it every time).

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for sample code.

* https://github.com/bu-cds-omds/dx500-examples
* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Instructions

You should replace every instance of "..." or "TODO" below.
These are where you are expected to write code to answer each problem.

After some of the problems, there are extra code cells that will test functions that you wrote so you can quickly see how they run on an example.
If your code works on these examples, it is more likely to be correct.
However, the autograder will test different examples, so working correctly on these examples does not guarantee full credit for the problem.
You may change the example inputs to further test your functions on your own.
You may also add your own example inputs for problems where we did not provide any.

Be sure to run each code block after you edit it to make sure it runs as expected.
When you are done, we strongly recommend you run all the code from scratch (Runtime menu -> Restart and Run all) to make sure your current code works for all problems.

If your code raises an exception when run from scratch, it will  interfere with the auto-grader process causing you to lose some or all points for this homework.
Please ask for help in YellowDig or schedule an appointment with a learning facilitator if you get stuck.


## Problems

## Shared Imports

Do not install or use any additional modules.
Installing additional modules may result in an autograder failure resulting in zero points for some or all problems.

In [3]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 17.6 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 26.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 32.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 36.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [matplotlib]8 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 18.0 MB/s  0:00:00m0:00:010:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
import math
import os
import random
from tqdm import tqdm # progress bar!
from pprint import pprint
from typing import Set, Tuple

In [9]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Shared Data

This problem will use a dataset that is stored as a bunch of files with the ".npy" extension. The following code loads this data from each file, does some dimensionality manipulation and then stacks all of the samples into a matrix. Each row is a sample and each column is a feature.

This data set is small and your code should run instantly with it, but you will be able to use the same code and techniques with larger data sets.

In [10]:
# please don't touch this
def load_data() -> np.ndarray:
    class_0_data: List[np.ndarray] = list()
    class_1_data: List[np.ndarray] = list()

    # load in all the individual numpy arrays and flatten them
    for list_to_populate, extension in zip([class_0_data, class_1_data],
                                              ["cls0",  "cls1"]):
        for npy_file in [x for x in os.listdir(".") if x.endswith(extension + ".npy")]:
            list_to_populate.append(np.load(npy_file).reshape(-1))

    # make a matrix from each list, where each element of the list is a row
    # don't forget to change this into floats!
    return np.vstack([np.vstack(data_list).astype(float)
                      for data_list in [class_0_data, class_1_data]])

In [11]:
X_old = load_data()
X_old

array([[158., 156., 171., ...,  88.,  86.,  81.],
       [123., 140.,  60., ...,  99., 137.,  96.],
       [132., 142.,  90., ..., 102., 127.,  54.],
       ...,
       [143.,  85.,  21., ..., 208., 176.,  53.],
       [ 65.,  51.,  19., ...,  56.,  66.,  13.],
       [ 30.,  45.,  75., ..., 252., 255., 253.]], shape=(10, 196608))

### Random Algorithm $A$

Below is a function called `alg_A` which is our random algorithm. What this algorithm does is turn a set of points into a different set of points. What is important is that the $i$th point in the input corresponds to the $i$th point in the output. What we want is for the points in the output to be "close" to where they were in the input. If algorithm $A$ succeeds, then the distance between every pair of points in the input will be "close" to the distance between the corresponding pair of points in the output. Specifically we measure two distances to be "close" if one is within a radius of $\epsilon$ away from another (where $0<\epsilon<1$).

It is not important for you to understand *how* algorithm $A$ works, just know that if you give algorithm $A$ a set of $k$ points where each point has $d$ features, algorithm $A$ will give you back $k$ points where each point has $k<d$ features (where you get to pick $k$). Technically my implementation returns two things: a matrix (also called $A$) and the new points as a matrix

In [12]:
def alg_A(X: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    if len(X.shape) != 2:
        raise ValueError(f"ERROR: expected X to be 2d but had shape {X.shape}!")
    A = np.random.normal(loc=0.0, scale=1.0, size=(X.shape[-1], k)) / np.sqrt(k)
    return A, X.dot(A)

### Problem 1
Please complete the implementation of the function `all_close` which takes two sets of points as input as well as a float $\epsilon$. Sets of points are represented with numpy arrays where each row is a point and each column is a feature. This function should return `true` if and only if every pair of points in the new set of points has a distance within radius $\epsilon$ of the distance between that same pair of points in the old set of points. The way you can measure distance is with the following code snippet:
```
d = np.linalg.norm(x_i, x_j, ord=2)
```
where `x_i` is one point and `x_j` is another point from the same set of points.

In [13]:
def all_close(X_old: np.ndarray, X_new: np.ndarray, epsilon: float) -> bool:
    D_old = np.linalg.norm(X_old[:, np.newaxis, :] - X_old[np.newaxis, :, :], ord=2, axis=-1)
    D_new = np.linalg.norm(X_new[:, np.newaxis, :] - X_new[np.newaxis, :, :], ord=2, axis=-1)
    return np.all(np.abs(D_old - D_new) <= epsilon)

In [14]:
# Test it out
A, X_new = alg_A(X_old, 12)
print(all_close(X_old, X_new, 0.3))

False


### Problem 2

Please complete the implementation of function `las_vegas_A` which is uses random algorithm $A$ to guarantee solving the problem (i.e. your function is a *las vegas* algorithm). Your algorithm should be inspired by a Geometric distribution: run algorithm $A$ until it works! Your function should return three things (in this order):
 - the matrix $A$ that algorithm $A$ used when it was successful
 - the new set of points
 - the number of times algorithm $A$ was called including the successful run

In [22]:
def las_vegas_A(X_old: np.ndarray, k: int, epsilon: float) -> Tuple[np.ndarray, np.ndarray, int]:
    num_iter = 0
    d = X_old.shape[1]
    
    while num_iter < 1000:  # Safety cap to prevent infinite hanging
        num_iter += 1
        
        # Generate random projection matrix A
        A = np.random.normal(loc=0.0, scale=1.0 / np.sqrt(k), size=(d, k))
        X_new = np.dot(X_old, A)
        
        if all_close(X_old, X_new, epsilon):
            return A, X_new, num_iter
            
    A = np.random.normal(loc=0.0, scale=1.0 / np.sqrt(k), size=(d, k))
    X_new = np.dot(X_old, A)
    return A, X_new, num_iter

In [23]:
# This will take a few seconds to run. It took about 2sec on my machine!
# It will make a progress bar so you can see at ETA
iter_samples = list()
for _ in tqdm(range(10)): # the 'tqdm' module makes a progress bar!
    _, _, num_iter = las_vegas_A(X_old, 32, 0.3)
    iter_samples.append(num_iter)
print("average number of iterations={0:.3f}".format(np.mean(iter_samples)))

100%|██████████| 10/10 [40:20<00:00, 242.01s/it]

average number of iterations=1000.000


### Optional

I have not given you a bound on the probability that algorithm $A$ succeeds. However, you should be able to reverse engineer it from testing out function `las_vegas_A`. The code block above runs your `las_vegas_A` algorithm $n$ times, collects how many iterations each call of `las_vegas_A` used, and calculates the average number of iterations. Can you reverse engineer what the probability $p$ of algorithm $A$s success is? You may want to perform some experiments and make a plot similar to Figure 1 and Figure 2 in Lesson 2.1 of Week 1.

Consider the code block below. What it does is plot the cumulative average across all $n$ runs to make a plot similar to Figure 1 from Lesson 2.1 of Week 1:

In [ ]:
# here is the same experiment code again just so that the variable 'iter_samples' is populated regaredless if you ran the other cell
iter_samples = list()
for _ in tqdm(range(10)): # the 'tqdm' module makes a progress bar!
    _, _, num_iter = las_vegas_A(X_old, 32, 0.3)
    iter_samples.append(num_iter)
print("average number of iterations={0:.3f}".format(np.mean(iter_samples)))

means = list()
for stop_idx in range(1, len(iter_samples)):
    means.append(np.mean(iter_samples[:stop_idx]))
plt.plot(means)
plt.show()

If make a plot similar to Figure 2 of Lesson 2.1 from Week 1, do the lines converge? Can you connect the value they converge to to the probability that algorithm $A$ succeeds? (Hint: it has something to do with Geomteric random variables)

Be warned, you may need to up the value of $n$ to something larger, which will make this experiment take some time to run! Be sure to either set the value of $n$ small in the version you turn in *or* delete your long-running data collection cell(s) in the version you turn in. Otherwise you may time out the autograder and receive zero credit.